# MLE — Train Arcade Game RL Agent on Colab GPU

Runs MAME headless on Colab + trains PPO/DQN with GPU acceleration.

**Requirements**: Upload your ROM zip (e.g. `galaga.zip`, `frogger.zip`)

In [ ]:
# 1. Install MAME and dependencies
!apt-get update -qq && apt-get install -y -qq mame > /dev/null 2>&1
!pip install -q gymnasium stable-baselines3 pytesseract pillow
!which mame && mame -version

In [ ]:
# 2. Clone MLE repo
!git clone https://github.com/lilwg/mle.git
%cd mle

In [ ]:
# 3. Upload ROM file
from google.colab import files
import os

uploaded = files.upload()  # Upload e.g. frogger.zip

os.makedirs('roms', exist_ok=True)
for fname in uploaded:
    os.rename(fname, f'roms/{fname}')
    print(f'ROM: roms/{fname}')

In [ ]:
# 4. Patch paths for Linux
import subprocess, os

mame_path = subprocess.run(['which', 'mame'], capture_output=True, text=True).stdout.strip()
print(f'MAME at: {mame_path}')

roms_path = os.path.abspath('roms')

for pyfile in ['mle_general.py', 'mle/console.py', 'validate_addrs.py',
               'find_score_ram.py', 'find_ram_addrs.py',
               'qbert_gym.py', 'qbert_bot.py', 'train_ppo.py']:
    if os.path.exists(pyfile):
        with open(pyfile) as f:
            code = f.read()
        code = code.replace('/opt/homebrew/bin/mame', mame_path)
        code = code.replace('/Users/pat/mame/roms', roms_path)
        with open(pyfile, 'w') as f:
            f.write(code)
        print(f'Patched {pyfile}')

In [ ]:
# 5. Quick test: verify MAME + MLE works
import sys
sys.path.insert(0, '.')
from mle import MameEnv

# Use the first ROM found
rom_name = [f[:-4] for f in os.listdir('roms') if f.endswith('.zip')][0]
print(f'Testing with ROM: {rom_name}')

env = MameEnv('roms', rom_name, {'_dummy': 0}, render=False, sound=False, throttle=False)
env.wait(100)
d = env.step()
print(f'MAME works! Got {len(d)} values')
env.close()
print('Ready to train!')

In [ ]:
# 6. Train!
# Change these settings as needed
GAME = rom_name      # uses the ROM you uploaded
TIMESTEPS = 5_000_000  # 5M steps
MODEL = 'ppo'          # ppo, dqn, or a2c

save_name = f'{GAME}_{MODEL}_{TIMESTEPS//1000}k'
print(f'Training {GAME} with {MODEL} for {TIMESTEPS} steps...')
print(f'Will save to {save_name}.zip')

!python3 mle_general.py {GAME} --model {MODEL} --timesteps {TIMESTEPS} --save {save_name}

In [ ]:
# 7. Download trained model
from google.colab import files
import glob
for f in glob.glob('*.zip'):
    if 'cheat' not in f:
        print(f'Downloading {f}...')
        files.download(f)